# Gold Price Prediction — Final Time-Series Forecasting Capstone
**Dataset:** Gold Price (2013–2023). **Target:** `Price` in USD as recorded by the source file.

## Problem statement
Build a reproducible workflow that diagnoses stationarity, compares classical time-series methods (AR, MA, ARMA, ARIMA, SARIMA) with baselines, and optionally evaluates RNN, LSTM, and GRU neural models.

### Research questions
1. Is the gold price level stationary?
2. Do first differences/log returns behave more like stationary series?
3. Which model performs best on a chronological holdout?
4. Do deep-learning models improve on simple statistical baselines?

**Important:** This is an educational exercise, not financial advice. Model rankings can change with the test window and parameters. Do not use random train/test splitting for time-series data.

In [ ]:
from pathlib import Path
import sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.data_utils import load_gold_data, regression_metrics
from src.stationarity import stationarity_report
from src.statistical_models import evaluate_statistical_models

pd.set_option("display.max_columns", 30)
warnings.filterwarnings("ignore")

## 1. Load and inspect data

In [ ]:
data_path = ROOT / "data" / "gold_price_2013_2023.csv"
df = load_gold_data(data_path)
print("Shape:", df.shape)
print("Date range:", df.index.min().date(), "to", df.index.max().date())
display(df.head())
display(df.tail())
display(df.describe().T)

In [ ]:
print("Missing values:")
display(df.isna().sum().to_frame("missing"))
print("Duplicate dates:", int(df.index.duplicated().sum()))
print("Date intervals (most common):")
display(df.index.to_series().diff().value_counts().head(10).to_frame("count"))

### Data preparation decisions
- The supplied CSV is reverse chronological; the loader sorts it oldest to newest.
- Comma separators in price columns are removed before numeric conversion.
- Invalid dates/prices and duplicate dates are handled by the loader.
- Volume and percentage-change fields are not required for this **univariate** forecast. They can be added later as exogenous features.
- The file actually spans **2013-01-02 through 2022-12-30**, despite the filename's 2013–2023 label.

## 2. Visualize the target

In [ ]:
y = df["Price"].astype(float)
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(y.index, y, linewidth=1)
ax.set(title="Historical Gold Price", xlabel="Date", ylabel="Price (USD, source units)")
ax.grid(True, alpha=.25)
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
axes[0].plot(y.index, y, linewidth=1)
axes[0].set_title("Price level")
axes[1].plot(y.index[1:], y.diff().dropna(), linewidth=.8)
axes[1].set_title("First difference")
axes[1].grid(True, alpha=.2)
plt.tight_layout()
plt.show()

## 3. Stationarity diagnosis — ADF and KPSS

- **ADF null hypothesis:** the series has a unit root (non-stationary). A small p-value supports stationarity.
- **KPSS null hypothesis:** the series is stationary around a constant. A small p-value supports non-stationarity.
- Using both helps avoid relying on one test alone. KPSS reports table-bound p-values (commonly 0.01 or 0.10), not always exact values.

In [ ]:
stationarity = stationarity_report(y)
display(stationarity)
stationarity.to_csv(ROOT / "reports" / "stationarity_summary.csv", index=False)

### Interpretation for this dataset
The price level's ADF p-value is about **0.761** and KPSS reports **0.01**: both results point to a non-stationary level series. For the first difference and log returns, ADF p-values are effectively zero and KPSS reports the upper table bound **0.10**, supporting stationarity for this sample. Therefore, use differencing for ARIMA-family modeling and fit AR/MA/ARMA to a stationary transformed series.

## 4. ACF / PACF of stationary transformation

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
stationary_series = y.diff().dropna()
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(stationary_series, lags=40, ax=axes[0])
plot_pacf(stationary_series, lags=40, ax=axes[1], method="ywm")
axes[0].set_title("ACF — first difference")
axes[1].set_title("PACF — first difference")
plt.tight_layout()
plt.show()

## 5. Chronological holdout and classical model comparison

The final 20% of observations are held out in date order. All models forecast the same test window. The baselines are persistence (last training price repeated) and drift. AR, MA, and ARMA are fit to first differences and then reconstructed to the price level. ARIMA and SARIMA are fit to price levels with differencing.

Metrics: **MAE**, **RMSE**, **MAPE** (lower is better). The test horizon is multi-step, so errors may grow with forecast horizon. Model fitting can take a few minutes.

In [ ]:
train, test, predictions, metrics = evaluate_statistical_models(
    y, test_size=0.20, seasonal_period=5
)
display(metrics)

In [ ]:
fig, ax = plt.subplots(figsize=(15, 6))
ax.plot(predictions.index, predictions["Actual"], label="Actual", linewidth=2)
for col in predictions.columns:
    if col != "Actual":
        ax.plot(predictions.index, predictions[col], label=col, alpha=.8, linewidth=1)
ax.set_title("Holdout: actual vs forecasts")
ax.set_xlabel("Date"); ax.set_ylabel("Price (USD, source units)")
ax.legend(ncol=2); ax.grid(True, alpha=.2)
plt.tight_layout()
plt.show()

In [ ]:
metrics.to_csv(ROOT / "reports" / "statistical_model_metrics.csv", index=False)
predictions.to_csv(ROOT / "reports" / "statistical_holdout_forecasts.csv")
print("Saved metrics and holdout predictions to reports/")

## 6. Optional deep learning — RNN, LSTM, GRU

This section requires TensorFlow and can be skipped if it is not installed. Install it in a compatible Python environment with `pip install tensorflow`. The scaler is fit only on the training segment to avoid leakage. Sequence windows preserve temporal order, and early stopping uses a chronological validation tail.

Neural networks are not automatically better for small univariate datasets. Compare their held-out metrics with the statistical models and naive baseline.

In [ ]:
try:
    from src.deep_learning import evaluate_neural_models
    neural_predictions, neural_metrics = evaluate_neural_models(
        y, test_size=0.20, lookback=30, epochs=25, batch_size=32
    )
    display(neural_metrics)
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(neural_predictions.index, neural_predictions["Actual"], label="Actual", linewidth=2)
    for col in ["RNN", "LSTM", "GRU"]:
        if col in neural_predictions:
            ax.plot(neural_predictions.index, neural_predictions[col], label=col)
    ax.legend(); ax.set_title("Neural models — holdout forecasts"); ax.grid(True, alpha=.2)
    plt.show()
    neural_metrics.to_csv(ROOT / "reports" / "neural_model_metrics.csv", index=False)
except ImportError as exc:
    print(exc)
    print("Skipping optional neural-network comparison.")

## 7. Model selection and conclusion

In [ ]:
if len(metrics.dropna(subset=["RMSE"])):
    display(metrics.dropna(subset=["RMSE"]).sort_values("RMSE").head(5))
    best = metrics.dropna(subset=["RMSE"]).sort_values("RMSE").iloc[0]
    print(f"Best statistical model on this holdout by RMSE: {best['Model']} (RMSE={best['RMSE']:.3f})")
else:
    print("No statistical model returned a valid metric. Review model errors and package versions.")

### Final conclusion template
Complete these points after running all cells:
1. **Stationarity:** report ADF and KPSS findings for level and differenced series.
2. **Best model:** name the model with the lowest holdout RMSE and report MAE/MAPE too.
3. **Baseline check:** confirm whether the chosen model beats the naive persistence forecast.
4. **Limitations:** single-variable historical data, structural changes, noisy markets, limited observations, possible trading-calendar gaps, and no macroeconomic/exogenous features.
5. **Future improvements:** rolling-origin backtesting, parameter tuning using validation data only, confidence intervals, exogenous variables (USD index, yields, inflation, ETF flows), and retraining on more recent data.

Do not claim a model is useful solely because it fits training data well. The chronological holdout comparison is the main evidence of forecast quality.